# Ruta del dinero en Bre-B

A partir de un extracto de transferencias Bre-B, este notebook:

1. califica cada cuenta de Cotrafa con reglas explicables (cuenta puente, muchos a uno, uno a muchos, ciclo, cuenta nueva, salida de madrugada);
2. toma una cuenta alertada y sigue el dinero: **de dónde viene, para dónde va y a dónde sigue**, salto por salto.

**Cómo usarlo**

1. Copie el archivo (`.xls`, `.xlsx` o `.csv`) en esta carpeta.
2. En la sección 1 cambie `ARCHIVO` y, si hace falta, los nombres de columnas.
3. Ejecute todas las celdas (*Run All*).

Por defecto lee `ejemplo_movimientos_breb.xls`, un archivo **sintético** creado con `GenerarEjemploBreB.ipynb`.

Para leer `.xls` se necesita `xlrd`: `pip install pandas numpy xlrd openpyxl`.

**Datos reales:** ejecútelo solo en el entorno de Cotrafa. Las salidas quedan en `salidas/`, que no se sube al repositorio. Antes de hacer commit del notebook, borre los resultados (*Clear All Outputs*).

## 1. Configuración

### 1.1 Archivo y parámetros

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

ARCHIVO = "ejemplo_movimientos_breb.xls"   # .xls, .xlsx o .csv
HOJA_MOVIMIENTOS = "movimientos"           # nombre de la hoja, o 0 para la primera
HOJA_CUENTAS = "cuentas"                   # hoja con fecha de apertura; None si no existe
NOMBRE_COTRAFA = "cotrafa"                 # texto que identifica a Cotrafa en las columnas de entidad
CARPETA_SALIDA = Path("salidas")

VENTANA_HORAS = 48     # el dinero que entra se busca en las salidas de las 48 h siguientes
MAX_SALTOS = 4         # saltos hacia adelante desde la entrada a la cuenta alertada
MAX_SALTOS_ATRAS = 2   # saltos hacia atrás
MONTO_MINIMO = 10_000  # por debajo de este monto se deja de seguir un hilo
SCORE_SEGUIR = 30      # el rastreo solo atraviesa cuentas de Cotrafa con este puntaje o más
SCORE_ALERTA = 60      # desde este puntaje la cuenta queda alertada

### 1.2 Nombres de columnas

A la izquierda el nombre que usa el notebook; a la derecha, el nombre de la columna en su archivo. No importa si está en mayúsculas o minúsculas.

- Si la fecha ya trae la hora, ponga `"hora": None`.
- `origen` y `destino` pueden ser números de cuenta o llaves Bre-B (seudonimizadas, si es posible).
- `COLUMNA_TIPOLOGIA` solo existe en el archivo de ejemplo; con datos reales déjela en `None`.

In [2]:
COLUMNAS = {
    "tx_id": "ID_TRANSACCION",
    "fecha": "FECHA",
    "hora": "HORA",
    "origen": "CUENTA_ORIGEN",
    "entidad_origen": "ENTIDAD_ORIGEN",
    "destino": "CUENTA_DESTINO",
    "entidad_destino": "ENTIDAD_DESTINO",
    "monto": "VALOR",
}
COLUMNAS_CUENTAS = {"cuenta": "CUENTA", "fecha_apertura": "FECHA_APERTURA"}
COLUMNA_TIPOLOGIA = "TIPOLOGIA"

### 1.3 Reglas y umbrales

Pesos y umbrales propuestos para calibrar con Riesgo y SARLAFT. El puntaje es la suma de puntos, con tope en 100.

In [3]:
REGLAS = pd.DataFrame([
    ("cuenta_puente", "Cuenta puente: entra y sale en menos de 1 hora", 40),
    ("ciclo", "Ciclo: el dinero vuelve a la cuenta", 30),
    ("muchos_a_uno", "Muchos a uno: 8 o más orígenes en 24 h", 25),
    ("uno_a_muchos", "Uno a muchos: 8 o más destinos en 24 h", 25),
    ("cuenta_nueva", "Cuenta con menos de 60 días", 20),
    ("salida_madrugada", "Salida de $1 millón o más entre 00:00 y 05:00", 15),
], columns=["regla", "descripcion", "puntos"]).set_index("regla")

PUENTE_MONTO_MIN = 500_000      # entradas mínimas de la semana para considerar cuenta puente
PUENTE_RATIO = (0.80, 1.05)     # salidas / entradas
PUENTE_MINUTOS = 60             # mediana de minutos entre entrada y salida
MUCHOS_CONTRAPARTES = 8         # orígenes o destinos distintos en 24 h
CUENTA_NUEVA_DIAS = 60
MADRUGADA_MONTO = 1_000_000
CICLO_MONTO_MIN = 1_000_000
CICLO_TOLERANCIA = 0.20         # cada paso del ciclo con monto ±20% del anterior
REGLAS

,descripcion,puntos
regla,,
cuenta_puente,Cuenta puente: entra y sale en menos de 1 hora,40
ciclo,Ciclo: el dinero vuelve a la cuenta,30
muchos_a_uno,Muchos a uno: 8 o más orígenes en 24 h,25
uno_a_muchos,Uno a muchos: 8 o más destinos en 24 h,25
cuenta_nueva,Cuenta con menos de 60 días,20
salida_madrugada,Salida de $1 millón o más entre 00:00 y 05:00,15


## 2. Cargar el archivo

In [4]:
ruta_archivo = Path(ARCHIVO)
if not ruta_archivo.exists():
    raise FileNotFoundError(f"No encuentro {ruta_archivo.resolve()}. Copie el archivo en esta carpeta o cambie ARCHIVO.")

if ruta_archivo.suffix.lower() == ".csv":
    libro, hojas = None, []
    crudo = pd.read_csv(ruta_archivo, dtype=str, sep=None, engine="python", encoding="utf-8-sig")
else:
    libro = pd.ExcelFile(ruta_archivo)  # .xls usa xlrd; .xlsx usa openpyxl
    hojas = libro.sheet_names
    crudo = libro.parse(HOJA_MOVIMIENTOS, dtype=str)

crudo.columns = crudo.columns.astype(str).str.strip().str.upper()
print("Hojas del libro:", hojas)
print(f"{len(crudo):,} filas · columnas: {list(crudo.columns)}")
crudo.head()

Hojas del libro: ['movimientos', 'cuentas']
6,047 filas · columnas: ['ID_TRANSACCION', 'FECHA', 'HORA', 'CUENTA_ORIGEN', 'ENTIDAD_ORIGEN', 'CUENTA_DESTINO', 'ENTIDAD_DESTINO', 'TIPO_LLAVE', 'VALOR', 'TIPOLOGIA']


,ID_TRANSACCION,FECHA,HORA,CUENTA_ORIGEN,ENTIDAD_ORIGEN,CUENTA_DESTINO,ENTIDAD_DESTINO,TIPO_LLAVE,VALOR,TIPOLOGIA
0,BRB-000001,2026-11-02 00:00:00,00:00:00,CTF-0377,COTRAFA,EXT-0043,ENTIDAD E,CELULAR,68500,normal
1,BRB-000002,2026-11-02 00:00:00,02:06:32,CTF-0177,COTRAFA,CTF-0118,COTRAFA,CELULAR,283600,normal
2,BRB-000003,2026-11-02 00:00:00,02:35:59,EXT-0217,ENTIDAD A,CTF-0317,COTRAFA,CORREO,123500,normal
3,BRB-000004,2026-11-02 00:00:00,02:44:18,CTF-0112,COTRAFA,EXT-0186,ENTIDAD B,ALFANUMERICA,174100,normal
4,BRB-000005,2026-11-02 00:00:00,03:09:41,EXT-0147,ENTIDAD B,CTF-0389,COTRAFA,DOCUMENTO,16900,normal


### 2.1 Revisar que estén las columnas

In [5]:
pedidas = {nombre: col.upper() for nombre, col in COLUMNAS.items() if col}
faltan = [f"{nombre} ('{col}')" for nombre, col in pedidas.items() if col not in crudo.columns]
if faltan:
    raise KeyError(f"Faltan columnas: {', '.join(faltan)}. Ajuste COLUMNAS en la sección 1.2. "
                   f"Columnas del archivo: {list(crudo.columns)}")

movs = crudo[list(pedidas.values())].rename(columns={col: nombre for nombre, col in pedidas.items()})
if COLUMNA_TIPOLOGIA and COLUMNA_TIPOLOGIA.upper() in crudo.columns:
    movs["tipologia"] = crudo[COLUMNA_TIPOLOGIA.upper()]
movs.head()

,tx_id,fecha,hora,origen,entidad_origen,destino,entidad_destino,monto,tipologia
0,BRB-000001,2026-11-02 00:00:00,00:00:00,CTF-0377,COTRAFA,EXT-0043,ENTIDAD E,68500,normal
1,BRB-000002,2026-11-02 00:00:00,02:06:32,CTF-0177,COTRAFA,CTF-0118,COTRAFA,283600,normal
2,BRB-000003,2026-11-02 00:00:00,02:35:59,EXT-0217,ENTIDAD A,CTF-0317,COTRAFA,123500,normal
3,BRB-000004,2026-11-02 00:00:00,02:44:18,CTF-0112,COTRAFA,EXT-0186,ENTIDAD B,174100,normal
4,BRB-000005,2026-11-02 00:00:00,03:09:41,EXT-0147,ENTIDAD B,CTF-0389,COTRAFA,16900,normal


## 3. Limpiar

### 3.1 Funciones de conversión

Los extractos traen montos como `1.234.567,89`, `$ 1,234,567.89` o números, y fechas como `04/11/2026` o `2026-11-04`. La hora puede venir como texto (`10:23:00`) o como fracción del día de Excel (`0.4326`).

In [6]:
def a_numero(valor):
    """'1.234.567,89', '$ 1,234,567.89' o 1234567.89 -> 1234567.89"""
    if pd.isna(valor):
        return np.nan
    texto = str(valor).replace("$", "").replace(" ", "").strip()
    if "," in texto and "." in texto:
        decimal = "," if texto.rfind(",") > texto.rfind(".") else "."
        texto = texto.replace("." if decimal == "," else ",", "").replace(decimal, ".")
    elif re.fullmatch(r"-?\d{1,3}([.,]\d{3})+", texto):
        texto = re.sub(r"[.,]", "", texto)  # solo separadores de miles
    else:
        texto = texto.replace(",", ".")
    return pd.to_numeric(texto, errors="coerce")


def a_fecha(serie):
    """Acepta fechas ISO (2026-11-04) y fechas con el día primero (04/11/2026)."""
    iso = pd.to_datetime(serie, format="ISO8601", errors="coerce")
    otras = pd.to_datetime(serie[iso.isna()], dayfirst=True, format="mixed", errors="coerce")
    return iso.fillna(otras)


def a_hora(serie):
    """'10:23:00' o 0.4326 (fracción del día) -> Timedelta."""
    como_numero = pd.to_numeric(serie, errors="coerce")
    texto = serie.astype(str).str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?)")[0]
    texto = texto.where(texto.str.count(":") == 2, texto + ":00")
    return pd.to_timedelta(como_numero, unit="D").fillna(pd.to_timedelta(texto, errors="coerce"))

### 3.2 Convertir tipos

In [7]:
movs["monto"] = movs.monto.map(a_numero)
movs["fecha"] = a_fecha(movs.fecha)
if "hora" in movs:
    movs["fecha"] = movs.fecha.dt.normalize() + a_hora(movs.pop("hora")).fillna(pd.Timedelta(0))
for col in ["tx_id", "origen", "destino", "entidad_origen", "entidad_destino"]:
    movs[col] = movs[col].astype(str).str.strip()
movs.dtypes

tx_id                         str
fecha              datetime64[us]
origen                        str
entidad_origen                str
destino                       str
entidad_destino               str
monto                       int64
tipologia                     str
dtype: object

### 3.3 Quitar filas que no sirven

Se quitan filas sin fecha, sin monto, con monto ≤ 0, transferencias de una cuenta a sí misma e identificadores repetidos (una misma transferencia cargada dos veces).

In [8]:
antes = len(movs)
problemas = {
    "sin fecha": movs.fecha.isna(),
    "sin monto o monto <= 0": ~(movs.monto > 0),
    "a sí misma": movs.origen == movs.destino,
    "id repetido": movs.tx_id.duplicated(),
}
for motivo, filas in problemas.items():
    print(f"{motivo}: {filas.sum():,}")
movs = movs[~pd.concat(problemas, axis=1).any(axis=1)].reset_index(drop=True)
print(f"Quedan {len(movs):,} de {antes:,}")

sin fecha: 0
sin monto o monto <= 0: 0
a sí misma: 0
id repetido: 0
Quedan 6,047 de 6,047


### 3.4 Identificar a Cotrafa

Una cuenta es de Cotrafa si su entidad contiene `NOMBRE_COTRAFA`. Cotrafa solo ve una transferencia cuando el origen o el destino es suyo; si alguna fila no cumple eso, se descarta.

In [9]:
movs["origen_cotrafa"] = movs.entidad_origen.str.contains(NOMBRE_COTRAFA, case=False, na=False)
movs["destino_cotrafa"] = movs.entidad_destino.str.contains(NOMBRE_COTRAFA, case=False, na=False)
movs.loc[movs.origen_cotrafa, "entidad_origen"] = "Cotrafa"
movs.loc[movs.destino_cotrafa, "entidad_destino"] = "Cotrafa"

sin_cotrafa = ~(movs.origen_cotrafa | movs.destino_cotrafa)
print(f"Filas sin cuenta de Cotrafa (descartadas): {sin_cotrafa.sum():,}")
movs = movs[~sin_cotrafa].sort_values("fecha", ignore_index=True)

print(f"Periodo: {movs.fecha.min():%Y-%m-%d %H:%M} a {movs.fecha.max():%Y-%m-%d %H:%M}")
pd.crosstab(movs.origen_cotrafa.map({True: "desde Cotrafa", False: "desde otra entidad"}),
            movs.destino_cotrafa.map({True: "hacia Cotrafa", False: "hacia otra entidad"}),
            values=movs.monto, aggfunc="sum", margins=True).round(0)

Filas sin cuenta de Cotrafa (descartadas): 0
Periodo: 2026-11-02 00:00 a 2026-11-08 23:59


destino_cotrafa,hacia Cotrafa,hacia otra entidad,All
origen_cotrafa,,,
desde Cotrafa,226676800.0,287068500.0,513745300
desde otra entidad,383865200.0,NaN,383865200
All,610542000.0,287068500.0,897610500


### 3.5 Entidad de cada cuenta

In [10]:
ENTIDAD = pd.concat([
    movs[["origen", "entidad_origen"]].set_axis(["cuenta", "entidad"], axis=1),
    movs[["destino", "entidad_destino"]].set_axis(["cuenta", "entidad"], axis=1),
]).drop_duplicates("cuenta").set_index("cuenta").entidad

CUENTAS_COTRAFA = sorted(ENTIDAD[ENTIDAD == "Cotrafa"].index)
print(f"{len(CUENTAS_COTRAFA):,} cuentas de Cotrafa · {(ENTIDAD != 'Cotrafa').sum():,} cuentas de otras entidades")
ENTIDAD.value_counts()

407 cuentas de Cotrafa · 307 cuentas de otras entidades


entidad
Cotrafa      407
ENTIDAD A     69
ENTIDAD E     63
ENTIDAD C     63
ENTIDAD B     60
ENTIDAD D     52
Name: count, dtype: int64

### 3.6 Antigüedad de las cuentas

Si el libro trae la hoja `HOJA_CUENTAS`, la antigüedad se calcula contra la última fecha del extracto. Si no la trae, la regla de cuenta nueva no se aplica.

In [11]:
FECHA_CORTE = movs.fecha.max().normalize()
if libro is not None and HOJA_CUENTAS in hojas:
    hoja = libro.parse(HOJA_CUENTAS, dtype=str)
    hoja.columns = hoja.columns.astype(str).str.strip().str.upper()
    cuentas = pd.DataFrame({
        "cuenta": hoja[COLUMNAS_CUENTAS["cuenta"].upper()].str.strip(),
        "fecha_apertura": a_fecha(hoja[COLUMNAS_CUENTAS["fecha_apertura"].upper()]),
    })
    cuentas["antiguedad_dias"] = (FECHA_CORTE - cuentas.fecha_apertura).dt.days
    print(f"{len(cuentas):,} cuentas con fecha de apertura")
else:
    cuentas = pd.DataFrame(columns=["cuenta", "fecha_apertura", "antiguedad_dias"])
    print("Sin hoja de cuentas: la regla 'cuenta_nueva' no se aplica.")
cuentas.describe()

407 cuentas con fecha de apertura


,fecha_apertura,antiguedad_dias
count,407,407.000000
mean,2020-04-09 13:05:27.272727,2403.454545
min,2013-03-01 00:00:00,14.000000
25%,2016-10-28 00:00:00,1174.000000
50%,2020-03-17 00:00:00,2427.000000
75%,2023-08-22 00:00:00,3663.000000
max,2026-10-25 00:00:00,5000.000000
std,NaN,1465.117537


## 4. Indicadores de flujo por cuenta

| Indicador | Qué mide |
|---|---|
| `ratio_paso` | salidas / entradas en pesos. Cerca de 1: el dinero no se queda. |
| `permanencia_min` | mediana de minutos entre la última entrada y cada salida. |
| `max_origenes_24h` | máximo de cuentas distintas que le enviaron dentro de 24 horas. |
| `max_destinos_24h` | máximo de cuentas distintas a las que envió dentro de 24 horas. |

### 4.1 Entradas y salidas de cada cuenta de Cotrafa

In [12]:
entradas = (movs[movs.destino_cotrafa]
            .rename(columns={"destino": "cuenta", "origen": "contraparte"}))
salidas = (movs[movs.origen_cotrafa]
           .rename(columns={"origen": "cuenta", "destino": "contraparte"}))

flujo = pd.DataFrame(index=pd.Index(CUENTAS_COTRAFA, name="cuenta"))
flujo = flujo.join(entradas.groupby("cuenta").agg(entradas_n=("tx_id", "size"), entradas_monto=("monto", "sum")))
flujo = flujo.join(salidas.groupby("cuenta").agg(salidas_n=("tx_id", "size"), salidas_monto=("monto", "sum")))
flujo = flujo.fillna(0)
flujo["ratio_paso"] = flujo.salidas_monto / flujo.entradas_monto.replace(0, np.nan)
flujo.sort_values("entradas_monto", ascending=False).head()

,entradas_n,entradas_monto,salidas_n,salidas_monto,ratio_paso
cuenta,,,,,
CTF-M01,5,24719000,10,23482900,0.949994
CTF-M02,5,14831400,5,14089900,0.950005
CTF-M04,1,9800000,11,9184800,0.937224
CTF-M06,2,6300000,1,3200000,0.507937
CTF-0071,13,4191000,6,346500,0.082677


### 4.2 Permanencia: cuánto tarda en salir lo que entra

Para cada salida se busca la entrada más reciente de la misma cuenta.

In [13]:
pares = pd.merge_asof(
    salidas[["cuenta", "fecha"]].sort_values("fecha"),
    entradas[["cuenta", "fecha"]].rename(columns={"fecha": "fecha_entrada"}).sort_values("fecha_entrada"),
    left_on="fecha", right_on="fecha_entrada", by="cuenta", direction="backward",
)
pares["espera_min"] = (pares.fecha - pares.fecha_entrada).dt.total_seconds() / 60
flujo["permanencia_min"] = pares.groupby("cuenta").espera_min.median()
flujo.permanencia_min.describe()

count     407.000000
mean      718.152129
std       574.562305
min         9.000000
25%       284.675000
50%       543.258333
75%      1074.466667
max      3950.633333
Name: permanencia_min, dtype: float64

### 4.3 Contrapartes distintas en 24 horas

In [14]:
def max_distintos_24h(grupo):
    fechas = grupo.fecha.to_numpy()
    otros = grupo.contraparte.to_numpy()
    fin = np.searchsorted(fechas, fechas + np.timedelta64(24, "h"), side="right")
    return max(len(set(otros[i:j])) for i, j in enumerate(fin))


flujo["max_origenes_24h"] = entradas.sort_values("fecha").groupby("cuenta")[["fecha", "contraparte"]].apply(max_distintos_24h)
flujo["max_destinos_24h"] = salidas.sort_values("fecha").groupby("cuenta")[["fecha", "contraparte"]].apply(max_distintos_24h)
flujo[["max_origenes_24h", "max_destinos_24h"]] = flujo[["max_origenes_24h", "max_destinos_24h"]].fillna(0).astype(int)
flujo[["max_origenes_24h", "max_destinos_24h"]].describe()

,max_origenes_24h,max_destinos_24h
count,407.000000,407.000000
mean,3.886978,3.329238
std,1.352612,1.195107
min,1.000000,1.000000
25%,3.000000,3.000000
50%,4.000000,3.000000
75%,5.000000,4.000000
max,14.000000,11.000000


### 4.4 Salidas de madrugada y antigüedad

In [15]:
madrugada = salidas[(salidas.fecha.dt.hour < 5) & (salidas.monto >= MADRUGADA_MONTO)]
flujo["salida_madrugada"] = flujo.index.isin(madrugada.cuenta)
flujo = flujo.join(cuentas.set_index("cuenta").antiguedad_dias)
print(f"Cuentas con salida de madrugada: {flujo.salida_madrugada.sum()}")
flujo.head()

Cuentas con salida de madrugada: 1


,entradas_n,entradas_monto,salidas_n,salidas_monto,ratio_paso,permanencia_min,max_origenes_24h,max_destinos_24h,salida_madrugada,antiguedad_dias
cuenta,,,,,,,,,,
CTF-0001,8,758300,8,1388900,1.831597,1266.383333,3,4,False,120
CTF-0002,11,1462000,14,1730100,1.183379,181.708333,4,4,False,454
CTF-0003,6,699800,10,1301200,1.859388,187.450000,4,3,False,1729
CTF-0004,9,972400,12,1712500,1.761107,274.741667,4,4,False,988
CTF-0005,13,1423300,11,1332000,0.935853,197.500000,4,4,False,4684


## 5. Ciclos

Un ciclo es una cadena A → B → … → A dentro de la ventana, con montos de `CICLO_MONTO_MIN` o más y parecidos entre sí. Indica que el dinero da una vuelta para aparentar movimientos legítimos.

In [16]:
grandes = movs[movs.monto >= CICLO_MONTO_MIN]
salidas_grandes = {c: list(g.itertuples()) for c, g in grandes.groupby("origen")}
VENTANA = pd.Timedelta(hours=VENTANA_HORAS)


def extender(camino, ciclos):
    """Agrega a `ciclos` todas las vueltas que cierran desde el último paso de `camino` (hasta 4 pasos)."""
    ultimo = camino[-1]
    for t in salidas_grandes.get(ultimo.destino, []):
        if not (ultimo.fecha < t.fecha <= camino[0].fecha + VENTANA):
            continue
        if abs(t.monto - ultimo.monto) > CICLO_TOLERANCIA * ultimo.monto:
            continue
        if t.destino == camino[0].origen:
            ciclos.append(camino + [t])
        elif len(camino) < 3 and t.destino not in {p.origen for p in camino}:
            extender(camino + [t], ciclos)

In [17]:
ciclos = []
for t in grandes.itertuples():
    extender([t], ciclos)

cuentas_en_ciclo = {c for ciclo in ciclos for t in ciclo for c in (t.origen, t.destino)}
for ciclo in ciclos:
    print(" → ".join(t.origen for t in ciclo) + f" → {ciclo[0].origen}",
          "|", " · ".join(f"{t.fecha:%d/%m %H:%M} ${t.monto:,.0f}" for t in ciclo))
print(f"{len(ciclos)} ciclos")

CTF-M06 → CTF-M07 → EXT-M15 → CTF-M06 | 05/11 11:00 $3,200,000 · 05/11 11:40 $3,100,000 · 05/11 14:40 $3,000,000
1 ciclos


## 6. Puntaje de riesgo por cuenta

### 6.1 Evaluar cada regla

In [18]:
reglas = pd.DataFrame(index=flujo.index)
reglas["cuenta_puente"] = ((flujo.entradas_monto >= PUENTE_MONTO_MIN)
                           & flujo.ratio_paso.between(*PUENTE_RATIO)
                           & (flujo.permanencia_min <= PUENTE_MINUTOS))
reglas["ciclo"] = flujo.index.isin(cuentas_en_ciclo)
reglas["muchos_a_uno"] = flujo.max_origenes_24h >= MUCHOS_CONTRAPARTES
reglas["uno_a_muchos"] = flujo.max_destinos_24h >= MUCHOS_CONTRAPARTES
reglas["cuenta_nueva"] = flujo.antiguedad_dias < CUENTA_NUEVA_DIAS
reglas["salida_madrugada"] = flujo.salida_madrugada
reglas.sum().rename("cuentas que cumplen")

cuenta_puente        6
ciclo                2
muchos_a_uno         3
uno_a_muchos         2
cuenta_nueva        20
salida_madrugada     1
Name: cuentas que cumplen, dtype: int64

### 6.2 Sumar puntos

In [19]:
riesgo = flujo.copy()
riesgo["score"] = (reglas.astype(int) * REGLAS.puntos).sum(axis=1).clip(upper=100).astype(int)
riesgo["nivel"] = pd.cut(riesgo.score, [-1, 29, 59, 79, 100], labels=["bajo", "medio", "alto", "crítico"])
riesgo["reglas"] = reglas.apply(lambda fila: list(fila.index[fila]), axis=1)
riesgo = riesgo.sort_values("score", ascending=False)

print(riesgo.nivel.value_counts().reindex(["crítico", "alto", "medio", "bajo"]).to_string())
riesgo[riesgo.score >= SCORE_SEGUIR][["score", "nivel", "reglas", "entradas_monto", "salidas_monto",
                                      "permanencia_min", "max_origenes_24h", "max_destinos_24h"]]

nivel
crítico      2
alto         4
medio        2
bajo       399


,score,nivel,reglas,entradas_monto,salidas_monto,permanencia_min,max_origenes_24h,max_destinos_24h
cuenta,,,,,,,,
CTF-M07,90,crítico,"[cuenta_puente, ciclo, cuenta_nueva]",3200000,3100000,40.000000,1,1
CTF-M04,85,crítico,"[cuenta_puente, uno_a_muchos, cuenta_nueva]",9800000,9184800,40.000000,1,11
CTF-M01,60,alto,"[cuenta_puente, cuenta_nueva]",24719000,23482900,9.000000,5,2
CTF-M02,60,alto,"[cuenta_puente, cuenta_nueva]",14831400,14089900,21.000000,1,1
CTF-M03,60,alto,"[muchos_a_uno, cuenta_nueva, salida_madrugada]",3782700,3669200,211.000000,14,1
CTF-M05,60,alto,"[cuenta_puente, cuenta_nueva]",739600,710000,17.000000,1,1
CTF-M06,50,medio,"[ciclo, cuenta_nueva]",6300000,3200000,60.000000,2,1
CTF-0387,40,medio,[cuenta_puente],1133700,936300,43.483333,3,3


## 7. Rastrear el dinero de una cuenta

**Hacia adelante (para dónde va).** Se parte de las entradas de la cuenta alertada. En cada cuenta se buscan sus salidas en las siguientes `VENTANA_HORAS` y se les asigna el dinero en orden de llegada: lo primero que entra es lo primero que sale. Una salida que ya se asignó a otro hilo no se cuenta dos veces.

**Hacia atrás (de dónde viene).** Si quien envió es una cuenta de Cotrafa con riesgo, se buscan sus entradas en las `VENTANA_HORAS` anteriores, de la más reciente a la más antigua.

**Dónde se detiene:**

- en una cuenta de Cotrafa sin riesgo (puntaje < `SCORE_SEGUIR`): el dinero se mezcla con el saldo propio del asociado y atribuirle sus gastos siguientes sería inventar la ruta;
- cuando sale hacia otra entidad, salvo que esa cuenta le vuelva a enviar a una cuenta de Cotrafa con riesgo (por ejemplo, en un ciclo).

### 7.1 Índices para buscar rápido

In [20]:
SCORE = riesgo.score.to_dict()
salidas_de = {c: list(g.sort_values("fecha").itertuples()) for c, g in movs.groupby("origen")}
entradas_a = {c: list(g.sort_values("fecha", ascending=False).itertuples()) for c, g in movs.groupby("destino")}


def es_cotrafa(cuenta):
    return ENTIDAD.get(cuenta) == "Cotrafa"


def se_puede_seguir(cuenta):
    return SCORE.get(cuenta, 0) >= SCORE_SEGUIR

### 7.2 Repartir un monto entre transferencias

Toma las transferencias candidatas en orden y le asigna a cada una lo que tenga libre, hasta agotar el monto. `usado` lleva cuánto de cada transferencia ya se asignó.

In [21]:
def repartir(monto, candidatas, usado):
    asignado = []
    for t in candidatas:
        libre = t.monto - usado.get(t.tx_id, 0)
        if libre <= 0:
            continue
        tomado = min(libre, monto)
        usado[t.tx_id] = usado.get(t.tx_id, 0) + tomado
        asignado.append((t, tomado))
        monto -= tomado
        if monto < MONTO_MINIMO:
            break
    return asignado

### 7.3 Candidatas en cada dirección

In [22]:
def salidas_siguientes(t):
    """Salidas de la cuenta que recibió t, dentro de la ventana, si el rastreo puede pasar por ahí."""
    cuenta = t.destino
    if es_cotrafa(cuenta) and not se_puede_seguir(cuenta):
        return []
    candidatas = [s for s in salidas_de.get(cuenta, []) if t.fecha < s.fecha <= t.fecha + VENTANA]
    if not es_cotrafa(cuenta):  # desde otra entidad solo se sigue si vuelve a una cuenta de riesgo
        candidatas = [s for s in candidatas if se_puede_seguir(s.destino)]
    return candidatas


def entradas_previas(t):
    """Entradas de la cuenta que envió t, dentro de la ventana anterior, si es de Cotrafa y tiene riesgo."""
    cuenta = t.origen
    if not (es_cotrafa(cuenta) and se_puede_seguir(cuenta)):
        return []
    return [e for e in entradas_a.get(cuenta, []) if t.fecha - VENTANA <= e.fecha < t.fecha]


def paso(t, salto, monto, direccion):
    return {"salto": salto, "direccion": direccion, "tx_id": t.tx_id, "fecha": t.fecha,
            "origen": t.origen, "destino": t.destino, "entidad_origen": t.entidad_origen,
            "entidad_destino": t.entidad_destino, "monto_tx": t.monto, "monto_rastreado": round(monto)}


COLUMNAS_PASO = ["salto", "direccion", "tx_id", "fecha", "origen", "destino",
                 "entidad_origen", "entidad_destino", "monto_tx", "monto_rastreado"]

### 7.4 Recorrer salto por salto

Las entradas a la cuenta alertada son el salto 1. Hacia adelante los saltos crecen (2, 3…); hacia atrás bajan (0, −1…).

In [23]:
def rastrear_adelante(semillas, max_saltos=MAX_SALTOS):
    usado = {}
    pasos = [paso(t, 1, t.monto, "entrada") for t in semillas]
    cola = [(t, t.monto, 1) for t in semillas]
    while cola:
        t, monto, salto = cola.pop(0)
        if salto >= max_saltos:
            continue
        for s, tomado in repartir(monto, salidas_siguientes(t), usado):
            pasos.append(paso(s, salto + 1, tomado, "adelante"))
            cola.append((s, tomado, salto + 1))
    return pd.DataFrame(pasos, columns=COLUMNAS_PASO)


def rastrear_atras(semillas, max_saltos=MAX_SALTOS_ATRAS):
    usado, pasos = {}, []
    cola = [(t, t.monto, 1) for t in semillas]
    while cola:
        t, monto, salto = cola.pop(0)
        if 1 - salto >= max_saltos:
            continue
        for e, tomado in repartir(monto, entradas_previas(t), usado):
            pasos.append(paso(e, salto - 1, tomado, "atras"))
            cola.append((e, tomado, salto - 1))
    return pd.DataFrame(pasos, columns=COLUMNAS_PASO)

### 7.5 Elegir la cuenta

Por defecto, la de mayor puntaje. Cambie `CUENTA` por cualquier cuenta de Cotrafa para ver su ruta.

In [24]:
CUENTA = riesgo.index[0]
semillas = list(movs[movs.destino == CUENTA].sort_values("fecha").itertuples())
print(f"{CUENTA} · score {SCORE[CUENTA]} · {len(semillas)} entradas")
riesgo.loc[[CUENTA], ["score", "reglas", "entradas_monto", "salidas_monto", "permanencia_min"]]

CTF-M07 · score 90 · 1 entradas


,score,reglas,entradas_monto,salidas_monto,permanencia_min
cuenta,,,,,
CTF-M07,90,"[cuenta_puente, ciclo, cuenta_nueva]",3200000,3100000,40.0


### 7.6 Para dónde va y a dónde sigue

In [25]:
adelante = rastrear_adelante(semillas)
adelante

,salto,direccion,tx_id,fecha,origen,destino,entidad_origen,entidad_destino,monto_tx,monto_rastreado
0,1,entrada,BRB-002877,2026-11-05 11:00:00,CTF-M06,CTF-M07,Cotrafa,Cotrafa,3200000,3200000
1,2,adelante,BRB-002922,2026-11-05 11:40:00,CTF-M07,EXT-M15,Cotrafa,ENTIDAD C,3100000,3100000
2,3,adelante,BRB-003149,2026-11-05 14:40:00,EXT-M15,CTF-M06,ENTIDAD C,Cotrafa,3000000,3000000


### 7.7 De dónde viene

In [26]:
atras = rastrear_atras(semillas)
print("Las entradas de la cuenta son el origen del dinero." if atras.empty else f"{len(atras)} pasos hacia atrás")
atras

1 pasos hacia atrás


,salto,direccion,tx_id,fecha,origen,destino,entidad_origen,entidad_destino,monto_tx,monto_rastreado
0,0,atras,BRB-002820,2026-11-05 10:00:00,EXT-M17,CTF-M06,ENTIDAD B,Cotrafa,3300000,3200000


### 7.8 Ruta completa

Si una misma transferencia aparece en varios hilos, se suma lo rastreado y se deja el salto más cercano.

In [27]:
def unir_ruta(atras, adelante):
    return (pd.concat([atras, adelante], ignore_index=True)
              .groupby("tx_id", as_index=False)
              .agg(salto=("salto", "min"), direccion=("direccion", "first"), fecha=("fecha", "first"),
                   origen=("origen", "first"), destino=("destino", "first"),
                   entidad_origen=("entidad_origen", "first"), entidad_destino=("entidad_destino", "first"),
                   monto_tx=("monto_tx", "first"), monto_rastreado=("monto_rastreado", "sum"))
              .sort_values(["salto", "fecha"], ignore_index=True))


ruta = unir_ruta(atras, adelante)
ruta

,tx_id,salto,direccion,fecha,origen,destino,entidad_origen,entidad_destino,monto_tx,monto_rastreado
0,BRB-002820,0,atras,2026-11-05 10:00:00,EXT-M17,CTF-M06,ENTIDAD B,Cotrafa,3300000,3200000
1,BRB-002877,1,entrada,2026-11-05 11:00:00,CTF-M06,CTF-M07,Cotrafa,Cotrafa,3200000,3200000
2,BRB-002922,2,adelante,2026-11-05 11:40:00,CTF-M07,EXT-M15,Cotrafa,ENTIDAD C,3100000,3100000
3,BRB-003149,3,adelante,2026-11-05 14:40:00,EXT-M15,CTF-M06,ENTIDAD C,Cotrafa,3000000,3000000


## 8. Resumen de la ruta

- **Entró:** lo que recibió la cuenta alertada.
- **Salió de Cotrafa:** lo que llegó a otras entidades y no volvió a Cotrafa dentro de la ruta.
- **Sigue en Cotrafa:** la diferencia; es lo que todavía se puede retener o congelar.

In [28]:
def resumir_ruta(cuenta, ruta):
    entro = ruta[(ruta.destino == cuenta) & (ruta.direccion == "entrada")].monto_rastreado.sum()
    adelante = ruta[ruta.direccion != "atras"]
    llega = adelante[adelante.entidad_destino != "Cotrafa"].groupby(["destino", "entidad_destino"]).monto_rastreado.sum()
    devuelve = adelante[adelante.entidad_origen != "Cotrafa"].groupby("origen").monto_rastreado.sum()
    neto = (llega - devuelve.reindex(llega.index.get_level_values(0)).fillna(0).to_numpy()).clip(lower=0)
    por_entidad = neto.groupby(level=1).sum().sort_values(ascending=False)
    return {"entro": float(entro), "salio_cotrafa": float(neto.sum()),
            "queda_en_cotrafa": float(max(entro - neto.sum(), 0)),
            "salio_por_entidad": {k: float(v) for k, v in por_entidad.items() if v > 0}}


resumir_ruta(CUENTA, ruta)

{'entro': 3200000.0, 'salio_cotrafa': 100000.0, 'queda_en_cotrafa': 3100000.0, 'salio_por_entidad': {'ENTIDAD C': 100000.0}}

## 9. Todas las cuentas alertadas

In [29]:
def ruta_de(cuenta, max_saltos=MAX_SALTOS):
    semillas = list(movs[movs.destino == cuenta].sort_values("fecha").itertuples())
    return unir_ruta(rastrear_atras(semillas), rastrear_adelante(semillas, max_saltos))


alertadas = riesgo[riesgo.score >= SCORE_ALERTA].index
rutas = {c: ruta_de(c) for c in alertadas}
resumen = pd.DataFrame({c: resumir_ruta(c, r) for c, r in rutas.items()}).T
resumen.insert(0, "score", riesgo.score.reindex(resumen.index))
resumen

,score,entro,salio_cotrafa,queda_en_cotrafa,salio_por_entidad
CTF-M07,90,3200000.0,100000.0,3100000.0,{'ENTIDAD C': 100000.0}
CTF-M04,85,9800000.0,4247300.0,5552700.0,"{'ENTIDAD D': 1565600.0, 'ENTIDAD C': 943100.0, 'ENTIDAD A': 896800.0, 'ENTIDAD B': 841800.0}"
CTF-M01,60,24719000.0,22741400.0,1977600.0,"{'ENTIDAD B': 14089900.0, 'ENTIDAD C': 8651500.0}"
CTF-M02,60,14831400.0,14089900.0,741500.0,{'ENTIDAD B': 14089900.0}
CTF-M03,60,3782700.0,3669200.0,113500.0,{'ENTIDAD E': 3669200.0}
CTF-M05,60,739600.0,710000.0,29600.0,{'ENTIDAD D': 710000.0}


## 10. Evaluación (solo con el archivo de ejemplo)

El archivo sintético trae la tipología de cada transferencia. Una cuenta mula es una cuenta de Cotrafa que recibe y envía dinero dentro de una tipología.

In [30]:
if "tipologia" in movs:
    fraude = movs[movs.tipologia.ne("normal")]
    mulas = set(fraude[fraude.origen_cotrafa].origen) & set(fraude[fraude.destino_cotrafa].destino)
    print(f"Cuentas mula: {len(mulas)} · alertadas (score ≥ {SCORE_ALERTA}): {len(alertadas)}")
    print(f"Detectadas: {len(mulas & set(alertadas))} · no detectadas: {sorted(mulas - set(alertadas))}")
    print(f"Falsos positivos: {sorted(set(alertadas) - mulas)}")
else:
    print("El archivo no trae tipologías: no hay contra qué evaluar.")

Cuentas mula: 7 · alertadas (score ≥ 60): 6
Detectadas: 6 · no detectadas: ['CTF-M06']
Falsos positivos: []


## 11. Exportar

En `salidas/`:

- `cuentas_riesgo.csv`: indicadores, reglas y puntaje de cada cuenta.
- `rutas_alertas.csv`: cada paso de la ruta de cada cuenta alertada.
- `ruta_dinero.json`: hasta 4 casos para el diagrama de `bre-b-sentinel.html`. Una cuenta que ya aparece en un caso no abre otro.

### 11.1 Preparar los casos para el diagrama

In [31]:
TITULOS = {"cuenta_puente": "Cuenta puente", "muchos_a_uno": "Muchos a uno", "uno_a_muchos": "Uno a muchos",
           "ciclo": "Ciclo", "salida_madrugada": "Salida de madrugada"}


def columnas_de(ruta):
    """Columna del diagrama de cada cuenta: el origen de un salto k va en k-1 y el destino en k."""
    columnas = {}
    for t in ruta.sort_values(["salto", "fecha"]).itertuples():
        columnas.setdefault(t.origen, t.salto - 1)
        columnas.setdefault(t.destino, t.salto)
    base = min(columnas.values())
    return {c: col - base for c, col in columnas.items()}, base


def tipo_nodo(c, cuenta, externas_que_siguen):
    if c == cuenta:
        return "alertada"
    if es_cotrafa(c):
        return "riesgo" if se_puede_seguir(c) else "cotrafa"
    return "externa" if c in externas_que_siguen else "sale"


def caso_json(cuenta, ruta):
    columnas, base = columnas_de(ruta)
    siguen = set(ruta[ruta.entidad_origen != "Cotrafa"].origen)
    fila = riesgo.loc[cuenta]
    return {
        "cuenta": cuenta, "score": int(fila.score), "nivel": str(fila.nivel),
        "titulo": " + ".join(TITULOS[r] for r in fila.reglas if r in TITULOS) or "Cuenta nueva",
        "reglas": [REGLAS.descripcion[r] for r in fila.reglas],
        "resumen": resumir_ruta(cuenta, ruta),
        "nodos": [{"id": c, "entidad": ENTIDAD.get(c), "col": col, "tipo": tipo_nodo(c, cuenta, siguen),
                   "score": int(SCORE.get(c, 0)) if es_cotrafa(c) else None} for c, col in columnas.items()],
        "enlaces": [{"tx": t.tx_id, "de": t.origen, "a": t.destino, "salto": int(t.salto - base),
                     "fecha": t.fecha.strftime("%Y-%m-%d %H:%M"), "monto_tx": float(t.monto_tx),
                     "monto": float(t.monto_rastreado)} for t in ruta.itertuples()],
    }

In [32]:
casos, cubiertas = [], set()
for c in alertadas:
    if c in cubiertas:
        continue
    caso = caso_json(c, ruta_de(c, max_saltos=3))
    casos.append(caso)
    cubiertas |= {n["id"] for n in caso["nodos"]}
    if len(casos) == 4:
        break
[(c["cuenta"], c["titulo"], len(c["nodos"])) for c in casos]

[('CTF-M07', 'Cuenta puente + Ciclo', 4), ('CTF-M04', 'Cuenta puente + Uno a muchos', 14), ('CTF-M01', 'Cuenta puente', 9), ('CTF-M03', 'Muchos a uno + Salida de madrugada', 16)]

### 11.2 Guardar

In [33]:
CARPETA_SALIDA.mkdir(exist_ok=True)
riesgo.assign(reglas=riesgo.reglas.str.join("|")).to_csv(CARPETA_SALIDA / "cuentas_riesgo.csv", encoding="utf-8-sig")
if rutas:
    pd.concat(rutas, names=["cuenta_alertada", "fila"]).reset_index(level=0).to_csv(
        CARPETA_SALIDA / "rutas_alertas.csv", index=False, encoding="utf-8-sig")
(CARPETA_SALIDA / "ruta_dinero.json").write_text(
    json.dumps({"archivo": ruta_archivo.name, "ventana_horas": VENTANA_HORAS, "casos": casos},
               ensure_ascii=False, indent=1), encoding="utf-8")
print("Guardado en", CARPETA_SALIDA.resolve())

Guardado en C:\CIENCIA DE DATOS\Cotrafa\Bre-b\salidas
